# RG3 12개 전체 새 학습 재현 검증


In [ ]:
from pathlib import Path
import json,hashlib,nbformat
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
root=Path.cwd();tables=root/'reports/tables'
s22=''.join(c.source for c in nbformat.read('notebooks/22_rg3_nested_signal_ensemble.ipynb',4).cells if c.cell_type=='code')
exec(s22[:s22.index('scores=[];thresholds=[];foldlogs=[]')].replace('n_jobs=-1','n_jobs=2'))
s42=''.join(c.source for c in nbformat.read('notebooks/42_rg3_rank_fusion_expert_ensembles.ipynb',4).cells if c.cell_type=='code')
expert_cache={}
exec(s42[s42.index('def expert('):s42.index('cache2={}')])
s154=nbformat.read('notebooks/154_rg3_gas_model_export.ipynb',4).cells[1].source
exec(s154[s154.index('def physical('):s154.index('def conditional_features(')])
F={k:pd.DataFrame(physical(d,k)) for k in ['thermal','pressure']}
controls={'thermal':['Barrel_Temperature_5','Max_Injection_Speed','Filling_Time','Hopper_Temperature'],'pressure':['Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3']}
s54=''.join(c.source for c in nbformat.read('notebooks/54_rg3_pdf_thermal_confirm_gates.ipynb',4).cells if c.cell_type=='code')
exec(s54[s54.index('def local_residual('):s54.index('memo={}')])
model_cache={};audit=[]
def fresh_scores(ref,q):
 key=(tuple(ref),tuple(q))
 if key in model_cache:return model_cache[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 overlap=len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q]))
 assert overlap==0
 audit.append(dict(train=len(ref),query=len(q),train_gas=int(y[ref].sum()),overlap=overlap))
 base=pair_predict(ref,q).max(axis=1)
 ex=expert(ref,q)
 outputs={'baseline':base,'expert':ex}
 z={}
 for kind in ['thermal','pressure']:
  tr=local_residual(ref,ref,kind);te=local_residual(ref,q,kind);z[kind]=te
  imp=SimpleImputer(keep_empty_features=True).fit(tr);tr=imp.transform(tr);te=imp.transform(te)
  arr=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   m=cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=2).fit(tr,y[ref])
   arr.append(m.predict_proba(te)[:,1])
  sc=RobustScaler().fit(tr[y[ref]==0])
  m=LogisticRegression(C=1.,class_weight='balanced',max_iter=2000,random_state=42).fit(sc.transform(tr),y[ref])
  arr.append(m.predict_proba(sc.transform(te))[:,1])
  outputs[kind]=np.mean(arr,axis=0)
 model_cache[key]=(outputs,z);return model_cache[key]
records=[];cutrows=[];first=None
for held in blocks:
 q=np.flatnonzero(groups.eq(held));ref=np.flatnonzero(~groups.eq(held))
 outer,zo=fresh_scores(ref,q)
 if first is None:first=(ref.copy(),q.copy(),{k:v.copy() for k,v in outer.items()})
 inner={k:np.full(len(ref),np.nan) for k in outer};zi=np.full((len(ref),6),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy()
  a,z=fresh_scores(ref[~mask],ref[mask])
  for k in inner:inner[k][mask]=a[k]
  zi[mask]=z['thermal']
 assert all(np.isfinite(v).all() for v in inner.values())
 normal=y[ref]==0
 cuts={k:float(np.quantile(inner[k][normal],quant,method='higher')) for k,quant in [('baseline',.99),('expert',.995),('thermal',.975),('pressure',.99)]}
 deltas=d.Barrel_Temperature_5__delta1.to_numpy()[ref]
 cuts['delta']=float(np.quantile(deltas[normal & np.isfinite(deltas)],.95,method='higher'))
 cuts['thermal_gate']=float(np.quantile(zi[normal,4],.05))
 b=outer['baseline']>cuts['baseline']
 e=(outer['expert']>cuts['expert'])&(d.Barrel_Temperature_5__delta1.to_numpy()[q]>cuts['delta'])
 th=(outer['thermal']>cuts['thermal'])&(zo['thermal'][:,4]<cuts['thermal_gate'])
 a=d.iloc[q]
 direction=((a.Max_Injection_Pressure__delta1.round(2)<0)&(a.Max_Injection_Speed__delta1.round(2)>0)&(a.Max_Switch_Over_Pressure__delta1.round(2)<a.Max_Injection_Pressure__delta1.round(2))).to_numpy()
 pressure=(outer['pressure']>cuts['pressure'])&direction
 pr=b|e|th|pressure
 cutrows.append(dict(block=held,**cuts))
 for j,idx in enumerate(q):
  row=dict(TimeStamp=d.TimeStamp.iloc[idx],block=held,actual=int(y[idx]),prediction=int(pr[j]),baseline=int(b[j]),expert=int(e[j]),thermal=int(th[j]),pressure=int(pressure[j]))
  row.update({k+'_score':float(outer[k][j]) for k in outer});records.append(row)
 pd.DataFrame(records).to_csv(tables/'rg3_193_fresh12_progress.csv',index=False)
 pd.DataFrame(cutrows).to_csv(tables/'rg3_193_fresh12_thresholds.csv',index=False)
 pd.DataFrame({'TimeStamp':d.TimeStamp.iloc[ref].to_numpy(),'actual':y[ref],'expert_score':inner['expert']}).to_csv(tables/('rg3_193_inner_'+held+'.csv'),index=False)
 print('Fresh all12 completed',held,flush=True)
pp=pd.DataFrame(records)
assert len(pp)==583 and pp.TimeStamp.is_unique and pp.actual.sum()==17
matches=None # No historical prediction CSV shipped in final release.
tn,fp,fn,tp=confusion_matrix(pp.actual,pp.prediction,labels=[0,1]).ravel()
print('Fresh counts',tp,fp,fn,tn,'previous match',matches,flush=True)
# Fresh deterministic replay plus heldout-label perturbation, no cached outputs.
ref,q,original=first
saved_y=y.copy();y=y.copy();y[q]=1-y[q]
cache.clear();paircache.clear();expert_cache.clear();model_cache.clear()
replayed,_=fresh_scores(ref,q)
delta={k:float(np.max(np.abs(replayed[k]-original[k]))) for k in original}
y=saved_y
assert all(v<1e-12 for v in delta.values())
pp.to_csv(tables/'rg3_193_fresh12_predictions.csv',index=False,encoding='utf-8-sig')
pp[pp.actual.eq(1)].to_csv(tables/'rg3_193_fresh12_all17_gas.csv',index=False,encoding='utf-8-sig')
pp[pp.actual.eq(0)&pp.prediction.eq(1)].to_csv(tables/'rg3_193_fresh12_false_positives.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audit).to_csv(tables/'rg3_193_fresh12_fit_audit.csv',index=False)
pd.DataFrame(cutrows).to_csv(tables/'rg3_193_fresh12_thresholds.csv',index=False)
summary=dict(classifiers=12,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=float(2*tp/(2*tp+fp+fn)),all583_predictions_match=matches,fit_calls=len(audit),all_fit_timestamp_overlap_zero=all(a['overlap']==0 for a in audit),heldout_label_flip_first_block_maxdiff=delta,input_sha256=hashlib.sha256((tables/'original_labeled_shot_features.csv').read_bytes()).hexdigest(),evaluation='existing 5-block development reproduction, not unseen independent validation',model_exported=False)
(tables/'rg3_193_fresh12_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
text='# RG3 12개 전체 새 학습 재현 검증\n\n'
text+='RF2+패턴RF4(seed42,2view×2mode)+열RF/ET/LR3+압력RF/ET/LR3. 이번 실행에서는 기존 예측을 학습 또는 결과 생성에 사용하지 않았다. 이전 CSV는 마지막 일치 검사에만 사용했다.\n\n'
text+=json.dumps(summary,ensure_ascii=False,indent=2)+'\n\n'
text+='학습과평가timestamp 및comparison block겹침없음. 임계값은각외부학습자료의내부구간제외정상점수로산정. 첫번째평가구간라벨반전후캐시전체비우고재학습한4개구성요소점수차이1e-12미만. 전처리Ridge4는분류기12개에포함하지않음.\n\n'
text+='## 주의\n이미검토한평가결과에서조합을골랐으므로새로운독립검증F1이아니다. 기존센서과거이력피처를재사용했다. 이력피처에는과거센서가포함되지만미래값/불량라벨을피처로넣지않는다. 이전평가센서이력조건까지새로운미래Run배포일반화로해석할수없다. 이번실행은저장된인과피처입력에서의전체12개재학습이며raw전처리처음부터재생성은아니다. 비라벨경보율미확인,저장추론모델미교체,commit/push없음.\n'
(root/'reports/rg3_193_fresh12_verification.md').write_text(text,encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
print(pp[pp.actual.eq(1)][['TimeStamp','prediction','baseline','expert','thermal','pressure']].to_string(index=False))

